In [ ]:
# 6.5 Comparação lado a lado: ANFIS vs Mamdani
from fuzzylab.fis import build_system as build_mamdani, run_inference as run_mamdani

mamdani_sim = build_mamdani()

print("## Comparação ANFIS vs Mamdani nos Cenários\n")
print("| Cenário | Variável | ANFIS | Mamdani | Δ |")
print("|---------|----------|-------|---------|---|")

for sr in scenario_results:
    mamdani_out = run_mamdani(mamdani_sim, sr.inputs)
    for key in ["sp", "wh", "ir", "bp"]:
        anfis_val = sr.anfis_outputs[key]
        mamdani_val = mamdani_out[key]
        delta = abs(anfis_val - mamdani_val)
        print(f"| {sr.name} | {key} | {anfis_val:.2f} | {mamdani_val:.2f} | {delta:.2f} |")

In [ ]:
# 6.4 Executar avaliação de cenários
scenario_results = evaluate_scenarios(result.model)

print("\n## Outputs ANFIS por Cenário\n")
print(scenarios_to_markdown(scenario_results))

In [ ]:
# 6.3 Avaliar cenários climáticos
print("## Cenários Climáticos\n")
print("Cenários predefinidos:")
for name, inputs in CLIMATE_SCENARIOS.items():
    print(f"  - {name}: T={inputs['Temperatura']}°C, U={inputs['Umidade']}%, " 
          f"C={inputs['Chuva']}mm, V={inputs['Vento']}km/h, ΔT={inputs['Delta T']}°C")

In [ ]:
# 6.2 Comparar ANFIS vs Mamdani (métricas por output)
metrics = compare_with_mamdani(result.model, X_val, y_val)

print("## Métricas ANFIS vs Mamdani\n")
print(metrics.to_markdown())
print()

# Verificar meta: R² > 0.90 para pelo menos 3 outputs
if metrics.meets_target(r2_threshold=0.90, min_outputs=3):
    print("\n✓ Meta atingida: R² > 0.90 para 3+ outputs")
else:
    print("\n✗ Meta NÃO atingida: R² < 0.90 para muitos outputs")

In [ ]:
# 6.1 Importar utilities de avaliação
from fuzzylab.anfis import (
    CLIMATE_SCENARIOS,
    compare_with_mamdani,
    compute_metrics,
    evaluate_scenarios,
    scenarios_to_markdown,
)

# Recarregar dataset para validação
X_val, y_val = load_dataset(DATASET_PATH)
print(f"Validation dataset: {X_val.shape[0]} samples")

# ANFIS Training Notebook

Este notebook treina o ANFIS para aproximar o comportamento do FIS Mamdani.

**Status:** Estrutura pronta, execução pendente de dataset maior (TASK-012).

## Índice
1. Setup e imports
2. Carregamento do dataset
3. Treinamento
4. Visualização de loss
5. Salvar pesos

In [ ]:
# 1. Setup e imports
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from fuzzylab.anfis import (
    AnfisNet,
    TrainingConfig,
    TrainingResult,
    create_dataloaders,
    initialize_from_mamdani,
    load_dataset,
    save_weights,
    train,
    train_with_lr_search,
)

# Configuração de reprodutibilidade
torch.manual_seed(42)

# Paths
PROJECT_ROOT = Path(".").resolve().parent
DATASET_PATH = PROJECT_ROOT / "data" / "raw" / "mamdani_dataset.csv"
WEIGHTS_PATH = PROJECT_ROOT / "data" / "models" / "anfis_weights.pt"

print(f"Dataset: {DATASET_PATH}")
print(f"Weights: {WEIGHTS_PATH}")

In [ ]:
# 2. Carregamento do dataset
X, y = load_dataset(DATASET_PATH)

print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"X range: [{X.min():.3f}, {X.max():.3f}]")
print(f"y range: [{y.min():.3f}, {y.max():.3f}]")

In [ ]:
# Criar DataLoaders
train_loader, val_loader = create_dataloaders(
    X, y,
    batch_size=64,
    val_split=0.2,
    seed=42,
)

n_train = sum(len(batch[0]) for batch in train_loader)
n_val = sum(len(batch[0]) for batch in val_loader)
print(f"Train samples: {n_train}")
print(f"Validation samples: {n_val}")

In [ ]:
# 3. Treinamento
# Criar modelo e inicializar com parâmetros do Mamdani
model = AnfisNet(n_inputs=5, n_mfs=7, n_outputs=4)
model = initialize_from_mamdani(model)

print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# Configuração de treinamento
config = TrainingConfig(
    lr=1e-3,
    batch_size=64,
    max_epochs=500,
    patience=10,
    seed=42,
    device="cpu",  # ou "cuda" se disponível
)

# Callback para mostrar progresso
def progress_callback(epoch: int, train_loss: float, val_loss: float) -> None:
    if epoch % 10 == 0 or epoch < 5:
        print(f"Epoch {epoch:3d}: train_loss={train_loss:.6f}, val_loss={val_loss:.6f}")

print(f"Training with lr={config.lr}...")

In [ ]:
# Executar treinamento
result = train(model, train_loader, val_loader, config, callback=progress_callback)

print(f"\nTraining completed:")
print(f"  Best epoch: {result.best_epoch}")
print(f"  Best val loss: {result.best_val_loss:.6f}")
print(f"  Stopped early: {result.stopped_early}")
print(f"  Total epochs: {len(result.train_losses)}")

In [ ]:
# 4. Visualização de loss
fig, ax = plt.subplots(figsize=(10, 6))

epochs = range(len(result.train_losses))
ax.plot(epochs, result.train_losses, label="Train Loss", alpha=0.8)
ax.plot(epochs, result.val_losses, label="Validation Loss", alpha=0.8)
ax.axvline(result.best_epoch, color="green", linestyle="--", label=f"Best Epoch ({result.best_epoch})")

ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("ANFIS Training Loss")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("figures/anfis_training_loss.png", dpi=150)
plt.show()

In [ ]:
# 5. Salvar pesos
save_weights(result.model, WEIGHTS_PATH)
print(f"Weights saved to: {WEIGHTS_PATH}")

## Experimento: Learning Rate Search

Teste com diferentes learning rates para encontrar o melhor.

In [ ]:
# Learning rate search (opcional)
# Descomente para executar

# def model_factory():
#     m = AnfisNet(n_inputs=5, n_mfs=7, n_outputs=4)
#     return initialize_from_mamdani(m)

# lr_config = TrainingConfig(
#     max_epochs=100,
#     patience=10,
#     lr_candidates=[1e-2, 1e-3, 1e-4],
# )

# best_result, best_lr = train_with_lr_search(
#     model_factory, train_loader, val_loader, lr_config, verbose=True
# )

# print(f"\nBest LR: {best_lr}")
# print(f"Best val loss: {best_result.best_val_loss:.6f}")

## Verificação de Meta

Critério de aceite TASK-014: MSE de validação abaixo de 0.05

In [ ]:
# Verificar meta
target_mse = 0.05
achieved = result.best_val_loss < target_mse

print(f"Target MSE: < {target_mse}")
print(f"Achieved MSE: {result.best_val_loss:.6f}")
print(f"Meta atingida: {'SIM' if achieved else 'NAO'}")

## 6. Validação ANFIS vs Mamdani (TASK-015)

Esta seção avalia quantitativamente a aproximação do ANFIS em relação ao FIS Mamdani.